In [0]:

from pyspark.sql.functions import col
import uuid

# 1. Thông số
topic_marketing = "bank_marketing"
checkpoint_marketing = "/Volumes/workspace/default/data_csv/checkpoints/cp_marketing"
marketing_bronze_path = "/Volumes/workspace/default/data_csv/marketing_bronze"

# 2. Đọc luồng từ Kafka
df_marketing_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "pkc-619z3.us-east1.gcp.confluent.cloud:9092") \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="ZWSEEXNPTALGFSAS" password="cfltHleM7FMu1XNTdcfRK34pwFF8GbCoO/JGkWCevJXYqCjp5MMhJPvsXp62e+tw";') \
    .option("subscribe", topic_marketing) \
    .option("startingOffsets", "earliest") \
    .load()

# 3. Ghi vào kho Bronze Marketing
query_marketing = df_marketing_raw.selectExpr("CAST(value AS STRING)") \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_marketing) \
    .trigger(availableNow=True) \
    .start(marketing_bronze_path)

query_marketing.awaitTermination()
print("✅ Dữ liệu Marketing đã nằm trong kho Bronze!")


In [0]:

from pyspark.sql.functions import from_json, when, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Định nghĩa Schema cho Marketing
marketing_schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("job", StringType(), True),
    StructField("marital", StringType(), True),
    StructField("education", StringType(), True),
    StructField("default", StringType(), True),
    StructField("balance", IntegerType(), True),
    StructField("housing", StringType(), True),
    StructField("loan", StringType(), True),
    StructField("contact", StringType(), True),
    StructField("day", IntegerType(), True),
    StructField("month", StringType(), True),
    StructField("duration", IntegerType(), True),
    StructField("campaign", IntegerType(), True),
    StructField("pdays", IntegerType(), True),
    StructField("previous", IntegerType(), True),
    StructField("poutcome", StringType(), True),
    StructField("y", StringType(), True) # Đây là cột target
])

# 2. Đọc từ Bronze và Parse
df_mkt_bronze = spark.read.format("delta").load(marketing_bronze_path)
df_mkt_parsed = df_mkt_bronze.withColumn("data", from_json(col("value"), marketing_schema)).select("data.*")

# 3. Chuyển đổi nhãn 'y' (yes=1, no=0) để dự đoán hành vi
df_mkt_silver = df_mkt_parsed.withColumn("label", when(col("y") == "yes", 1.0).otherwise(0.0))

# 4. Lưu vào Silver Marketing
marketing_silver_path = "/Volumes/workspace/default/data_csv/marketing_silver"
df_mkt_silver.write.format("delta").mode("overwrite").save(marketing_silver_path)
print("✅ Tầng Silver Marketing hoàn tất!")



In [0]:

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# 1. Đọc Silver Marketing
df_mkt_gold_raw = spark.read.format("delta").load(marketing_silver_path)

# 2. Pipeline xử lý đặc trưng (Mixed Data: Age + Job + Education)
cat_cols = ["job", "marital", "education", "housing", "loan", "contact"]
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in cat_cols]

num_cols = ["age", "balance", "duration", "campaign"]
assembler = VectorAssembler(inputCols=num_cols + [c+"_idx" for c in cat_cols], outputCol="features")

# 3. Train Model dự đoán hành vi (Target: label)
rf_mkt = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)
pipeline_mkt = Pipeline(stages=indexers + [assembler] + [rf_mkt])

# Chia dữ liệu và Train
train_mkt, test_mkt = df_mkt_gold_raw.randomSplit([0.8, 0.2])
model_mkt = pipeline_mkt.fit(train_mkt)

# 4. Dự đoán
predictions_mkt = model_mkt.transform(test_mkt)
predictions_mkt.select("age", "job", "prediction", "label").show(10)
print("✅ Đã hoàn thành mô hình Dự đoán hành vi khách hàng trên kênh số!")


In [0]:

from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

def print_marketing_metrics(predictions, model_name):
    # 1. Khởi tạo các bộ đánh giá
    multi_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
    binary_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction")
    
    # 2. Tính toán từng chỉ số
    acc = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "accuracy"})
    prec = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedPrecision"})
    rec = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "weightedRecall"})
    auc = binary_evaluator.evaluate(predictions) # Mặc định là areaUnderROC
    
    print(f"📊 CHỈ SỐ CỦA MÔ HÌNH DỰ ĐOÁN HÀNH VI ({model_name}):")
    print(f"--------------------------------------------------")
    print(f"- Accuracy (Độ chính xác tổng thể): {acc:.4f}")
    print(f"- Precision (Khả năng dự đoán đúng khách chốt đơn): {prec:.4f}")
    print(f"- Recall (Khả năng không bỏ sót khách tiềm năng): {rec:.4f}")
    print(f"- AUC (Khả năng phân biệt khách 'Yes' và 'No'): {auc:.4f}")
    print(f"--------------------------------------------------\n")

# 3. Chạy hàm đánh giá trên kết quả dự đoán của Dataset 2
print_marketing_metrics(predictions_mkt, "RANDOM FOREST - BANK MARKETING")
# Lưu kết quả dự đoán Marketing
predictions_mkt.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/data_csv/marketing_results")
print("✅ Đã lưu kết quả dự đoán Marketing.")
